In [3]:
import re
import time
import json
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    WebDriverException,
    NoSuchElementException,
    TimeoutException
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ============================ CONFIG ============================
EXCEL_IN = "bidassis_tenderresults_27-08-25.xlsx"
EXCEL_OUT = "tender_scraped_bid_27-08.xlsx"
COOKIES_FILE = "cookies.json"
LIMIT = None   # set to None to process all rows
BASE_RESULT = "https://nexizo.ai/app/discover/tenders/result/"
BASE_DETAIL = "https://nexizo.ai/app/discover/tenders/detail/"  # fallback
PAGE_LOAD_TIMEOUT = 25

UUID_REGEX = re.compile(
    r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$",
    re.IGNORECASE
)

# ====================== READ INPUT EXCEL ========================
df = pd.read_excel(EXCEL_IN)

# Choose the column robustly: prefer URL, else Tender ID, else first column
if "Tender ID URL" in df.columns:
    items = df["Tender ID URL"].astype(str).str.strip().tolist()
elif "Tender ID" in df.columns:
    items = df["Tender ID"].astype(str).str.strip().tolist()
elif "TenderID" in df.columns:
    items = df["TenderID"].astype(str).str.strip().tolist()
else:
    items = df.iloc[:, 0].astype(str).str.strip().tolist()

if LIMIT:
    items = items[:LIMIT]

# ================ SELENIUM: LOGIN BYPASS ========================
driver = webdriver.Chrome()
wait = WebDriverWait(driver, PAGE_LOAD_TIMEOUT)

driver.get("https://nexizo.ai")   # open the site domain first
time.sleep(2)

# Load cookies.json
with open(COOKIES_FILE, "r", encoding="utf-8") as f:
    cookies = json.load(f)

for cookie in cookies:
    cookie.pop("sameSite", None)   # remove unsupported key
    cookie.pop("sameSitePolicy", None)

    # If expirationDate exists, rename to expiry
    if "expirationDate" in cookie:
        try:
            cookie["expiry"] = int(cookie.pop("expirationDate"))
        except Exception:
            cookie.pop("expirationDate", None)

    try:
        driver.add_cookie(cookie)
    except WebDriverException:
        # Retry without domain if it mismatches
        cookie.pop("domain", None)
        try:
            driver.add_cookie(cookie)
        except Exception:
            pass

# Refresh to apply cookies
driver.refresh()
time.sleep(3)

# ====================== SCRAPING HELPERS ========================
def ensure_loaded():
    """Wait until the body exists and the app has rendered something."""
    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    except TimeoutException:
        pass
    # small settle
    time.sleep(1.5)

def build_url(item: str) -> str:
    """Accepts URL, UUID, or ID; prefers /result/, falls back to /detail/."""
    s = item.strip()
    if s.lower().startswith("http"):
        return s
    # UUID -> result URL
    if UUID_REGEX.match(s):
        return BASE_RESULT + s
    # Not URL, not UUID: try as result first; if that fails we’ll fallback
    return BASE_RESULT + s

def page_has_result_stages() -> bool:
    """Return True if 'Result Stages' section is present."""
    try:
        driver.find_element(By.XPATH, "//h3[contains(normalize-space(),'Result Stages')]")
        return True
    except NoSuchElementException:
        return False

def get_text_or_blank(by, selector, ctx=None):
    """Safe text getter."""
    try:
        if ctx is None:
            el = driver.find_element(by, selector)
        else:
            el = ctx.find_element(by, selector)
        return el.text.strip()
    except NoSuchElementException:
        return ""

def scrape_one(url_or_id: str) -> dict:
    """
    Load the page (prefer /result/), scrape:
      - TenderID from page
      - Status: 'Technical Bid' if Result Stages exists, else 'Awarded'
      - Bidders: list of 'Name — Status'
    Fallback to /detail/ if the first attempt yields nothing recognizable.
    """
    # First attempt
    url_try = build_url(url_or_id)

    def do_scrape(at_url: str):
        driver.get(at_url)
        ensure_loaded()

        # TENDER ID from page
        tender_id_on_page = get_text_or_blank(
            By.XPATH, "//h3[normalize-space()='Tender Id']/following-sibling::*[1]"
        )

        # STATUS
        status = "Technical Bid" if page_has_result_stages() else "Awarded"

        # BIDDERS (repeatable blocks)
        bidders = []
        blocks = driver.find_elements(
            By.XPATH,
            "//div[contains(@class,'bidder-list')]//div[contains(@class,'detail-list-wrap')]"
        )
        for blk in blocks:
            # Name can be in <a> or <span> immediately following the 'Bidder Name' label
            name = ""
            try:
                name = blk.find_element(
                    By.XPATH,
                    ".//h3[normalize-space()='Bidder Name']/following-sibling::*[1]"
                ).text.strip()
            except NoSuchElementException:
                pass

            # Per-block bidder status
            bstatus = ""
            try:
                bstatus = blk.find_element(
                    By.XPATH,
                    ".//h3[contains(., 'Status')]/following-sibling::span[contains(@class,'value')]"
                ).text.strip()
            except NoSuchElementException:
                pass

            if name:
                bidders.append(f"{name} — {bstatus}" if bstatus else name)

        return tender_id_on_page, status, bidders

    tender_id, status, bidders = do_scrape(url_try)

    # Fallback to /detail/ if we couldn't get a tender id and no bidders
    if not tender_id and not bidders and not url_try.startswith(BASE_DETAIL):
        if url_or_id.lower().startswith("http"):
            # If the original was a URL and didn't work, try swapping result->detail
            alt = url_or_id.replace("/result/", "/detail/")
        else:
            alt = BASE_DETAIL + url_or_id
        tender_id, status, bidders = do_scrape(alt)
        final_url = alt
    else:
        final_url = url_try

    # As a last resort, use the ID from list if page didn’t show one
    if not tender_id:
        # sometimes the visible "Tender Id" label may be slightly different
        tender_id = get_text_or_blank(By.XPATH, "//h3[contains(.,'Tender Id')]/following-sibling::*[1]") or ""

    if not tender_id:
        # fallback to source value
        tender_id = url_or_id

    return {
        "URL": final_url,
        "TenderID": tender_id,
        "Status": status,
        "Bidders_List": bidders  # used to expand Bidder1..N later
    }

# ===================== RUN SCRAPING LOOP ========================
rows = []
for item in items:
    print(f"🔎 Scraping: {item}")
    try:
        rows.append(scrape_one(item))
    except Exception as e:
        print(f"❌ Error scraping {item}: {e}")
        rows.append({
            "URL": item if str(item).lower().startswith("http") else "",
            "TenderID": str(item),
            "Status": "",
            "Bidders_List": []
        })

# ================== SHAPE OUTPUT COLUMNS ========================
# Expand Bidder1..BidderN
max_bidders = max((len(r["Bidders_List"]) for r in rows), default=0)
records = []
for r in rows:
    rec = {
        "URL": r["URL"],
        "TenderID": r["TenderID"],
        "Status": r["Status"]
    }
    for i in range(max_bidders):
        rec[f"Bidder{i+1}"] = r["Bidders_List"][i] if i < len(r["Bidders_List"]) else ""
    records.append(rec)

out_df = pd.DataFrame(records)
out_df.to_excel(EXCEL_OUT, index=False)
print(f"🎉 Done! Saved: {EXCEL_OUT}")

# Optional: quick peek
print(out_df.head(min(5, len(out_df))))

driver.quit()


🔎 Scraping: 51d866b6-3226-4aed-b5df-bc0044e3a13b
🔎 Scraping: 52f2de54-158a-4619-83b5-b74814313fe3
🔎 Scraping: 9549ddf0-2269-4839-9f1a-36addb8da1ef
🔎 Scraping: 41209fe5-6d5c-4965-8c32-37fff17e505b
🔎 Scraping: 22510bfc-6bdd-43d4-acdf-553e0f2f68fa
🔎 Scraping: b440f071-14a1-46c8-99bf-4ff07cf75ccb
🔎 Scraping: 7eeaf965-60a1-4e79-9442-551d9004e3cd
🔎 Scraping: fde7f96d-7274-4c6d-a989-3807cd2377b7
🔎 Scraping: 6aa65872-4bd5-446e-8c51-8888a372b6fe
🔎 Scraping: 076cd713-51cc-4bc1-bb60-0cedfb3b8cc7
🔎 Scraping: e77dbe63-5692-4c14-8d01-90d7b05c18c2
🔎 Scraping: f1cf4d97-aa81-4aed-9204-c462ef37bd81
🔎 Scraping: a1c2d414-a23f-48e9-8078-ef49c3018558
🔎 Scraping: 56c7324e-d834-4661-b9b1-e356763744a8
🔎 Scraping: d7b11b04-195f-48cb-9427-312d46f9766e
🔎 Scraping: c6534545-473e-4c55-9f7d-b1070a095ca0
🔎 Scraping: 014e50d6-5481-4b97-bf18-009ead986cac
🔎 Scraping: 8bd70001-2f76-4bac-ab0a-e57052601412
🔎 Scraping: 1bbb0265-fc55-45a6-8f70-d16ac7593783
🔎 Scraping: b9215df4-ec85-4a62-b88e-9fdee93d026d
🔎 Scraping: a145bbee